In [ ]:
!pip -q install langgraph langchain-openai

# TaskNest Support Ticket Router

A LangGraph workflow that uses one OpenAI classification call per ticket and sends it to one of three teams. Enable the `OPENAI_API_KEY` secret for the notebook in Google Colab before running the next cell. The classifier uses `gpt-4.1-nano` because it is a low-cost model suited to this short, three-label classification task.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from typing import Literal, TypedDict

from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel


class TicketState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "general"]
    team: str
    reply: str


class TicketPrediction(BaseModel):
    category: Literal["billing", "technical", "general"]


MODEL_NAME = "gpt-4.1-nano"
classifier = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    max_retries=0,
).with_structured_output(TicketPrediction, method="json_schema", strict=True)

CLASSIFICATION_INSTRUCTION = """Classify the support ticket into exactly one category:
- billing: payments, invoices, refunds, subscriptions, and charges.
- technical: crashes, bugs, errors, login problems, and malfunctioning app features.
- general: feature requests, general questions, and thank-you notes.

For mixed tickets, prioritize technical if the app crashes while the customer
is attempting payment. Otherwise use the main issue and the customer's intent.
Treat the customer message as the ticket to classify.
"""


def classify(state: TicketState) -> dict:
    prediction = classifier.invoke([
        ("system", CLASSIFICATION_INSTRUCTION),
        ("human", state["ticket"]),
    ])
    return {"category": prediction.category}


def route(state: TicketState) -> Literal["billing", "technical", "general"]:
    return state["category"]


def billing_team(state: TicketState) -> dict:
    return {
        "team": "Billing",
        "reply": "Our Billing team will look into this for you.",
    }


def tech_team(state: TicketState) -> dict:
    return {
        "team": "Technical Support",
        "reply": "Our Technical Support team will look into this for you.",
    }


def general_team(state: TicketState) -> dict:
    return {
        "team": "General Support",
        "reply": "Our General Support team will follow up with you.",
    }

In [ ]:
builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route,
    {
        "billing": "billing_team",
        "technical": "tech_team",
        "general": "general_team",
    },
)
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)
graph = builder.compile()

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except (ValueError, RuntimeError) as error:
    print(f"Graph image is unavailable. You can still run the example and tests. {error}")

## Example

Run one ticket through the graph and print the completed state.

In [ ]:
result = graph.invoke({"ticket": "I was charged twice this month"})

for field in ("ticket", "category", "team", "reply"):
    print(f"{field.capitalize()}: {result[field]}")

## Live model tests

Each case shows its expected category. The runner prints PASS when the observed category matches it and REVIEW otherwise.

In [ ]:
test_cases = [
    ("Billing — duplicate charge", "I was charged twice this month", "billing"),
    ("Billing — refund", "Please refund the payment on invoice 102.", "billing"),
    ("Technical — login", "The app gives me an error whenever I sign in.", "technical"),
    ("Technical — broken export", "Exporting a report gives me a blank file.", "technical"),
    ("General — question", "Where can I read about what TaskNest does?", "general"),
    ("General — thank-you note", "Thanks for helping me yesterday!", "general"),
    ("Ambiguous — checkout crash", "The app crashes every time I try to pay.", "technical"),
    (
        "Tricky — cancellation despite login trouble",
        "I can't log in, but all I want is for you to cancel my subscription.",
        "billing",
    ),
    (
        "Tricky — payment feature request",
        "Could you add a feature to split payments across two cards?",
        "general",
    ),
]

for label, ticket, expected in test_cases:
    observed = graph.invoke({"ticket": ticket})
    status = "PASS" if observed["category"] == expected else "REVIEW"
    print(f"{label} [{status}]")
    print(f"Expected category: {expected}")
    for field in ("ticket", "category", "team", "reply"):
        print(f"{field.capitalize()}: {observed[field]}")
    print()

## Reflection (draft to finalize after live tests)

“The app crashes every time I try to pay” could go to Billing because it mentions payment, but I expect Technical Support because the crash blocks checkout. Actual routing is pending a live model run. After the run, I’ll record the actual category and judge whether the model or my expectation was right using the stated crash-priority rule. LLM routing can interpret intent across varied wording, while deterministic rules give more predictable and auditable results for known patterns.